# 02 - Data Cleaning

## Online Retail Sales & Customer Analysis

This notebook prepares the raw transaction data for exploratory and customer-level analysis.

Cleaning decisions are based on the project's analytical objectives and the findings identified during the data-understanding stage. Rather than automatically removing unusual values, each cleaning step is applied according to the meaning and intended use of the data.

## 1. Load Data

In [1]:
import pandas as pd

In [ ]:
# Load DataSet
df = pd.read_excel("../OnlineRetail.xlsx")

In [4]:
df.shape

(541909, 8)

In [5]:
df.isna().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

In [6]:
df.describe()

,Quantity,InvoiceDate,UnitPrice,CustomerID
count,541909.000000,541909,541909.000000,406829.000000
mean,9.552250,2011-07-04 13:34:57.156386,4.611114,15287.690570
min,-80995.000000,2010-12-01 08:26:00,-11062.060000,12346.000000
25%,1.000000,2011-03-28 11:34:00,1.250000,13953.000000
50%,3.000000,2011-07-19 17:17:00,2.080000,15152.000000
75%,10.000000,2011-10-19 11:27:00,4.130000,16791.000000
max,80995.000000,2011-12-09 12:50:00,38970.000000,18287.000000
std,218.081158,NaN,96.759853,1713.600303


## 2. Create Revenue

In [7]:
df["Revenue"] = df["Quantity"] * df["UnitPrice"]

In [8]:
df["Revenue"].describe()

count    541909.000000
mean         17.987795
std         378.810824
min     -168469.600000
25%           3.400000
50%           9.750000
75%          17.400000
max      168469.600000
Name: Revenue, dtype: float64

In [ ]:
df.nlargest(10,"Revenue")[["InvoiceNo", "StockCode", "Description", "Quantity", "UnitPrice", "CustomerID", "Revenue"]]

In [ ]:
df.nsmallest(10, "Revenue")[["InvoiceNo", "StockCode", "Description", "Quantity", "UnitPrice", "CustomerID", "Revenue"]]

### 3. Remove Exact Duplicates

Exact duplicate rows were identified using all available columns. These rows contain identical transaction information and could cause the same transaction to be counted more than once.

A total of **5,268 duplicate copies** were identified and removed, while one occurrence of each duplicated record was retained.

After removing duplicates, the dataset contains **536,641 transaction lines**.

In [25]:
duplicated_count = df.duplicated().sum()
duplicated_count

np.int64(5268)

In [ ]:
df[df.duplicated() == True].head(20)

In [ ]:
df[df.duplicated(keep=False)].sort_values(
    ["InvoiceNo", "StockCode"]
).head(20)

In [28]:
df = df.drop_duplicates()

In [30]:
df.count()

InvoiceNo      536641
StockCode      536641
Description    535187
Quantity       536641
InvoiceDate    536641
UnitPrice      536641
CustomerID     401604
Country        536641
Revenue        536641
dtype: int64

## 4. Handle Missing Values

### 4.1 Missing Description

Description → keep missing values; only ~0.27% of cleaned rows affected.

### 4.2 Missing CustomerID

CustomerID → keep missing values; important for transaction-level analysis but unavailable for identified-customer analysis.

In [ ]:
df[df["Description"].isna()][["InvoiceNo", "StockCode", "Description", "Quantity", "UnitPrice", "CustomerID", "Country", "Revenue"]
].head(20)

## 5. Handle Data Types

   ### 5.1 CustomerID

In [34]:
df["CustomerID"] = df["CustomerID"].astype("Int64")

In [49]:
df["CustomerID"].dtype

Int64Dtype()

### 6. Handle Unusual Values

### 6.1 Negative Quantities

Negative quantities were investigated during the data-understanding stage. They are strongly associated with invoices beginning with `C`, suggesting that they represent cancellation, return, or other negative transactions.

These rows were **retained** because removing them would prevent negative transactions from reducing net revenue.

However, their treatment may differ depending on the analytical objective. For example, negative transactions may need to be handled differently when analyzing net revenue versus customer purchasing frequency.

### 6.2 Zero UnitPrice

Investigation showed multiple types of records, so we retain them.

### 6.3 Negative UnitPrice

Two transactions were found with a negative `UnitPrice`. Both corresponded to the special transaction `Adjust bad debt` rather than a normal product sale.

These records were **retained** in the cleaned dataset because they represent a special transaction rather than an ordinary product sale. Their treatment can be considered separately when performing product-level sales analysis.

In [50]:
df[df["UnitPrice"] < 0]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue
299983,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,<NA>,United Kingdom,-11062.06
299984,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,<NA>,United Kingdom,-11062.06


### 6.4 Extreme UnitPrice / Revenue

Extreme values were investigated and found to include genuine high-value products as well as special transactions; no universal threshold was justified, so they are retained.

## 7. StockCode and Description Consistency
 
Inspection showed that some StockCode values are associated with multiple descriptions, including operational notes such as incorrectly coded or marked items. Therefore, StockCode and Description are not treated as a strict one-to-one mapping, and descriptions are not automatically filled or replaced based on StockCode.

In [36]:
df["StockCode"].value_counts().head(20)

StockCode
85123A    2301
22423     2192
85099B    2156
47566     1720
20725     1626
84879     1489
22720     1469
22197     1468
21212     1367
22383     1328
20727     1323
22457     1272
23203     1260
POST      1256
22386     1245
22469     1232
22960     1221
21931     1211
22086     1194
22411     1192
Name: count, dtype: int64

In [41]:
stock_description_counts = (
    df.groupby("StockCode")["Description"]
    .nunique()
    .sort_values(ascending=False)
)

stock_description_counts.head(20)

StockCode
20713     8
23084     7
21830     6
85175     6
72807A    5
85172     5
23343     5
21181     5
23131     5
85185B    4
85123A    4
46000S    4
72802A    4
72807B    4
22837     4
23196     4
22812     4
22501     4
22121     4
21823     4
Name: Description, dtype: int64

In [43]:
df[df["StockCode"] == 20713]["Description"].unique()

array(['JUMBO BAG OWLS', nan, 'wrongly marked. 23343 in box',
       'wrongly coded-23343', 'found', 'Found', 'wrongly marked 23343',
       'Marked as 23343', 'wrongly coded 23343'], dtype=object)

These were identified as special/non-standard transactions and retained in the main dataset because their exclusion depends on the analytical objective.

In [44]:
df[df["Description"].isin([
    "POSTAGE",
    "AMAZON FEE",
    "Manual",
    "Discount",
    "Adjust bad debt"
])][["StockCode", "Description"]].drop_duplicates()

,StockCode,Description
45,POST,POSTAGE
141,D,Discount
2239,M,Manual
14514,AMAZONFEE,AMAZON FEE
40383,m,Manual
299982,B,Adjust bad debt


In [45]:
special_codes = ["POST", "D", "M", "m", "AMAZONFEE", "B"]

df[df["StockCode"].isin(special_codes)].shape

(1937, 9)

## 8. Final Validation

In [46]:
df.shape

(536641, 9)

In [38]:
df.dtypes

InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
UnitPrice             float64
CustomerID              Int64
Country                   str
Revenue               float64
dtype: object

In [39]:
df.isna().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135037
Country             0
Revenue             0
dtype: int64

In [47]:
df.duplicated().sum()

np.int64(0)

## 9. Cleaning Summary

The cleaning process was guided by the findings from the data-understanding stage rather than by automatic removal of unusual values.

The following actions were performed:

* Added `Revenue` as `Quantity × UnitPrice`.
* Removed **5,268 exact duplicate rows**.
* Converted `CustomerID` from `float64` to nullable `Int64`.
* Retained missing `CustomerID` values because they are relevant for transaction-level analysis but require special treatment in customer-level analysis.
* Retained missing `Description` values because only a small proportion of records are affected and the rows themselves may still contain valid transaction information.
* Retained negative quantities because they may represent returns or cancellations and are necessary for calculating net revenue.
* Retained zero, negative, and unusually high `UnitPrice` values because the investigation showed that they can represent different types of legitimate or special transactions.
* Retained special transaction types such as postage, manual adjustments, discounts, and fees in the main dataset. These can be excluded later when a specific analysis requires product-only transactions.
* Verified that the final dataset contains **536,641 rows and 9 columns** and **no exact duplicate rows**.

The cleaned dataset is now ready for exploratory data analysis.